<a href="https://colab.research.google.com/github/Dadi-Lakshmi-Harsha/GPU_Programminig/blob/main/Gpu_assignment3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%%writefile assigment3.cu
#include <iostream>
#include <cstdlib>
#include <ctime>
#include <cmath>
#include <cuda_runtime.h>
#include <cuda_fp16.h>
#include <mma.h>
using namespace std;
using namespace nvcuda;

#define CHECK(call) \
do { \
    cudaError_t e = call; \
    if (e != cudaSuccess) { \
        cout << "CUDA error: " << cudaGetErrorString(e) << endl; \
        exit(1); \
    } \
} while(0)

const int N = 64;
const int TILE = 16;

__global__ void tensorMul(const half *a, const half *b, float *c, int n)
{
    int tr = blockIdx.y;
    int tc = blockIdx.x;

    wmma::fragment<wmma::matrix_a,16,16,16,half,wmma::row_major> fa;
    wmma::fragment<wmma::matrix_b,16,16,16,half,wmma::row_major> fb;
    wmma::fragment<wmma::accumulator,16,16,16,float> fc;

    wmma::fill_fragment(fc, 0.0f);

    for (int k = 0; k < n; k += TILE)
    {
        const half *pa = a + tr * TILE * n + k;
        const half *pb = b + k * n + tc * TILE;

        wmma::load_matrix_sync(fa, pa, n);
        wmma::load_matrix_sync(fb, pb, n);
        wmma::mma_sync(fc, fa, fb, fc);
    }

    float *pc = c + tr * TILE * n + tc * TILE;
    wmma::store_matrix_sync(pc, fc, n, wmma::mem_row_major);
}

void cpuMul(const float *a, const float *b, float *c, int n)
{
    for (int i = 0; i < n; i++)
    {
        for (int j = 0; j < n; j++)
        {
            float sum = 0.0f;

            for (int k = 0; k < n; k++)
                sum += a[i * n + k] * b[k * n + j];

            c[i * n + j] = sum;
        }
    }
}

int main()
{
    srand(time(nullptr));

    int total = N * N;

    float *ha = new float[total];
    float *hb = new float[total];
    float *hc = new float[total];
    float *ref = new float[total];

    half *fa = new half[total];
    half *fb = new half[total];

    for (int i = 0; i < total; i++)
    {
        ha[i] = static_cast<float>(rand() % 5 + 1);
        hb[i] = static_cast<float>(rand() % 5 + 1);

        fa[i] = __float2half(ha[i]);
        fb[i] = __float2half(hb[i]);
    }

    half *da;
    half *db;
    float *dc;

    CHECK(cudaMalloc(&da, total * sizeof(half)));
    CHECK(cudaMalloc(&db, total * sizeof(half)));
    CHECK(cudaMalloc(&dc, total * sizeof(float)));

    CHECK(cudaMemcpy(da, fa, total * sizeof(half),
                     cudaMemcpyHostToDevice));

    CHECK(cudaMemcpy(db, fb, total * sizeof(half),
                     cudaMemcpyHostToDevice));

    dim3 grid(N / TILE, N / TILE);
    dim3 block(32);

    cudaEvent_t start, stop;

    CHECK(cudaEventCreate(&start));
    CHECK(cudaEventCreate(&stop));

    CHECK(cudaEventRecord(start));

    tensorMul<<<grid, block>>>(da, db, dc, N);

    CHECK(cudaGetLastError());
    CHECK(cudaEventRecord(stop));
    CHECK(cudaEventSynchronize(stop));

    float time = 0.0f;
    CHECK(cudaEventElapsedTime(&time, start, stop));

    CHECK(cudaMemcpy(hc, dc, total * sizeof(float),
                     cudaMemcpyDeviceToHost));

    cpuMul(ha, hb, ref, N);

    bool ok = true;
    float maxErr = 0.0f;

    for (int i = 0; i < total; i++)
    {
        float err = fabs(hc[i] - ref[i]);

        if (err > maxErr)
            maxErr = err;

        if (err > 0.1f)
        {
            ok = false;
            break;
        }
    }

    cout << "Matrix size: " << N << " x " << N << endl;
    cout << "Tile size: " << TILE << " x " << TILE << endl;
    cout << "Grid: " << grid.x << " x " << grid.y << endl;
    cout << "Threads/block: " << block.x << endl;

    if (ok)
        cout << "correct output" << endl;
    else
        cout << "wrong answer" << endl;

    cout << "maximum error: " << maxErr << endl;
    cout << "execution time: " << time << " ms" << endl;

    CHECK(cudaEventDestroy(start));
    CHECK(cudaEventDestroy(stop));

    CHECK(cudaFree(da));
    CHECK(cudaFree(db));
    CHECK(cudaFree(dc));

    delete[] ha;
    delete[] hb;
    delete[] hc;
    delete[] ref;
    delete[] fa;
    delete[] fb;

    return 0;
}

Writing assigment3.cu


In [2]:
!nvcc -arch=sm_75 -o assigment3 assigment3.cu
!./assigment3

Matrix size: 64 x 64
Tile size: 16 x 16
Grid: 4 x 4
Threads/block: 32
correct output
maximum error: 0
execution time: 0.102528 ms
